# 🤖 Software Engineering RAG Chatbot — Professional Rebuild

**Hybrid multilingual semantic assistant for Software Engineering / Génie Logiciel**

This notebook implements a modern Retrieval-Augmented Generation (RAG) pipeline:

```
User question (FR or EN)
        │
        ▼
  Language detection  ──► optional FR→EN translation
        │
        ▼
   Preprocessing  (normalization, accents, whitespace)
        │
        ▼
  Multilingual embedding  (paraphrase-multilingual-MiniLM-L12-v2)
        │
        ▼
   FAISS semantic search  (Top-K candidates)
        │
        ▼
  CrossEncoder reranking  (ms-marco-MiniLM-L-6-v2)
        │
        ▼
  Context construction + conversational memory
        │
        ▼
   FLAN-T5 generation
        │
        ▼
  Optional EN→FR translation back
        │
        ▼
       Gradio chat UI
```

### Why this architecture

| Limitation of the previous notebook                | How we fix it                                            |
|----------------------------------------------------|----------------------------------------------------------|
| Hardcoded dataset inside Python code               | External CSV files loaded and merged                     |
| Brute-force cosine similarity                      | FAISS index (scales to millions of vectors)              |
| Returns only one candidate                         | Top-K retrieval + CrossEncoder reranking                 |
| No conversational memory                           | Rolling chat history fed into the generator              |
| No real preprocessing                              | Modular cleaner: lowercase, accents, punct, whitespace   |
| English-only T5 + French users                     | Multilingual embeddings + optional translation pipeline  |
| No evaluation                                      | Retrieval@k, MRR, semantic similarity, confusion table   |
| Monolithic notebook                                | 12 modular, independently runnable sections              |

### Stack

`pandas` · `sentence-transformers` · `faiss-cpu` · `transformers` · `scikit-learn` · `gradio`

> **Runtime**: works on Google Colab (CPU is enough; GPU recommended for the generator). Total install + first run ≈ 5 minutes.


## 1. Setup — Install and import dependencies

We pin lightweight versions known to work together on Colab. If a package is already present we keep the existing version (faster startup).


In [ ]:
# --- Section 1.1 : Install (idempotent, quiet) ----------------------------
import importlib, subprocess, sys

REQUIRED = {
    "pandas":                "pandas",
    "numpy":                 "numpy",
    "sklearn":               "scikit-learn",
    "sentence_transformers": "sentence-transformers",
    "faiss":                 "faiss-cpu",
    "transformers":          "transformers",
    "torch":                 "torch",
    "gradio":                "gradio",
    "langdetect":            "langdetect",
    "unidecode":             "unidecode",
}

missing = []
for mod, pkg in REQUIRED.items():
    try:
        importlib.import_module(mod)
    except ImportError:
        missing.append(pkg)

if missing:
    print(f"Installing: {missing}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
    print("Done. If you see an import error in the next cell on Colab, just re-run this cell.")
else:
    print("All dependencies already installed.")


In [ ]:
# --- Section 1.2 : Imports + global config --------------------------------
import os, re, json, time, random, warnings
from pathlib import Path
from typing import List, Dict, Tuple, Optional

import numpy as np
import pandas as pd
import torch

from sentence_transformers import SentenceTransformer, CrossEncoder
import faiss

warnings.filterwarnings("ignore")
random.seed(42); np.random.seed(42); torch.manual_seed(42)

DEVICE       = "cuda" if torch.cuda.is_available() else "cpu"
DATA_DIR     = Path("data");      DATA_DIR.mkdir(exist_ok=True)
INDEX_DIR    = Path("indexes");   INDEX_DIR.mkdir(exist_ok=True)
ARTIFACT_DIR = Path("artifacts"); ARTIFACT_DIR.mkdir(exist_ok=True)

print(f"Device : {DEVICE}")
print(f"Torch  : {torch.__version__}")


## 2. Sample external CSV datasets

In a real deployment you would point the loader at full dumps from sources such as:

* **Stack Exchange data dump** — `softwareengineering.stackexchange.com` ([archive.org/details/stackexchange](https://archive.org/details/stackexchange))
* **Kaggle StackSample** — 60k programming Q&A
* **Hugging Face datasets** — `KK04/LogicInference_OA`, `mteb/stackexchange-clustering`, `flax-sentence-embeddings/stackexchange_xml`
* Your own course notes converted to CSV

To keep the notebook self-contained and reproducible, **Section 2 writes a representative seed dataset to disk as multiple CSVs**, organized by topic. The pipeline below will load *any* CSV found in `data/` with the schema `question,answer,category,language` — you can drop additional files in and re-run.


In [ ]:
# --- Section 2.1 : Seed CSV datasets (Software Engineering) ---------------
# Each row: question, answer, category, language ('en' or 'fr')
# Concise, textbook-style entries covering the main software engineering topics.
# In production, replace with full Stack Exchange / Kaggle dumps using the SAME schema.

seed_data = {
    "agile_scrum.csv": [
        ("What is Agile methodology?",
         "Agile is an iterative software development approach that delivers working software in short cycles called sprints, embraces changing requirements, and emphasizes collaboration between cross-functional teams and customers.",
         "agile", "en"),
        ("What is Scrum?",
         "Scrum is an Agile framework structured around fixed-length iterations called sprints (usually 1-4 weeks), three roles (Product Owner, Scrum Master, Development Team), and ceremonies such as sprint planning, daily standup, sprint review and retrospective.",
         "agile", "en"),
        ("What is the role of a Scrum Master?",
         "The Scrum Master is a servant-leader who helps the team follow Scrum practices, removes impediments, shields the team from external interruptions, and facilitates the Scrum ceremonies.",
         "agile", "en"),
        ("What is a user story?",
         "A user story is a short, informal description of a feature written from the user's perspective, typically in the form: 'As a <role>, I want <goal> so that <benefit>'. It is the basic unit of work in many Agile methods.",
         "agile", "en"),
        ("What is a sprint retrospective?",
         "A sprint retrospective is a Scrum ceremony held at the end of each sprint where the team inspects how the last sprint went and identifies improvements for the next one, focusing on people, relationships, process and tools.",
         "agile", "en"),
        ("What is the difference between Scrum and Kanban?",
         "Scrum uses fixed-length sprints with committed scope and defined roles, while Kanban is a continuous flow system that visualizes work on a board and limits work-in-progress (WIP) without prescribed roles or time-boxes.",
         "agile", "en"),
        ("Qu'est-ce que la methode Agile ?",
         "La methode Agile est une approche iterative du developpement logiciel qui livre des increments fonctionnels dans des cycles courts appeles sprints, accepte les changements de besoins et privilegie la collaboration entre l'equipe et le client.",
         "agile", "fr"),
        ("Qu'est-ce que Scrum ?",
         "Scrum est un cadre Agile structure autour de sprints de duree fixe (1 a 4 semaines), de trois roles (Product Owner, Scrum Master, equipe de developpement) et de ceremonies : planification, melee quotidienne, revue et retrospective.",
         "agile", "fr"),
    ],
    "uml.csv": [
        ("What is UML?",
         "UML (Unified Modeling Language) is a standardized graphical modeling language used to specify, visualize and document the artifacts of an object-oriented software system. It defines around 14 diagram types split into structural and behavioral diagrams.",
         "uml", "en"),
        ("What is a class diagram?",
         "A class diagram is a UML structural diagram that shows the classes of a system, their attributes, methods and the static relationships (association, aggregation, composition, inheritance, dependency) between them.",
         "uml", "en"),
        ("What is a sequence diagram?",
         "A sequence diagram is a UML behavioral diagram that shows how objects interact over time through messages exchanged between lifelines. It emphasizes the temporal ordering of messages.",
         "uml", "en"),
        ("What is the difference between aggregation and composition?",
         "Both express a whole-part relationship. Aggregation is a weak relationship: the part can exist independently of the whole. Composition is strong: the lifecycle of the part is bound to that of the whole, so destroying the whole destroys the parts.",
         "uml", "en"),
        ("What is a use case diagram?",
         "A use case diagram is a UML behavioral diagram that represents the functional requirements of a system through actors (external entities) and use cases (units of functionality), connected by associations.",
         "uml", "en"),
        ("Qu'est-ce qu'un diagramme de classes ?",
         "Un diagramme de classes est un diagramme UML structurel qui decrit les classes du systeme, leurs attributs, leurs methodes et les relations statiques (association, agregation, composition, heritage, dependance) entre elles.",
         "uml", "fr"),
        ("Difference entre agregation et composition ?",
         "L'agregation et la composition expriment toutes deux une relation tout-partie. L'agregation est faible : la partie peut exister independamment du tout. La composition est forte : la partie est detruite avec le tout.",
         "uml", "fr"),
    ],
    "oop.csv": [
        ("What is object-oriented programming?",
         "Object-oriented programming (OOP) is a paradigm based on objects that combine data (attributes) and behavior (methods). Its four pillars are encapsulation, inheritance, polymorphism and abstraction.",
         "oop", "en"),
        ("What is encapsulation?",
         "Encapsulation is the OOP principle of bundling data and the methods that operate on it inside a class, and restricting direct access to internal state through visibility modifiers (private, protected, public).",
         "oop", "en"),
        ("What is polymorphism?",
         "Polymorphism is the OOP ability for a single interface or method name to behave differently depending on the actual object type. Common forms are method overriding (runtime) and method overloading (compile time).",
         "oop", "en"),
        ("What is inheritance?",
         "Inheritance is an OOP mechanism by which a subclass derives attributes and methods from a parent class, allowing code reuse and the modeling of 'is-a' relationships.",
         "oop", "en"),
        ("What is an abstract class?",
         "An abstract class is a class that cannot be instantiated directly and is meant to be subclassed. It typically defines abstract methods that subclasses must implement, plus optionally concrete shared behavior.",
         "oop", "en"),
        ("What are SOLID principles?",
         "SOLID is a set of five OOP design principles: Single Responsibility, Open/Closed, Liskov Substitution, Interface Segregation and Dependency Inversion. Together they promote maintainable, extensible code.",
         "oop", "en"),
        ("What is the difference between an interface and an abstract class?",
         "An interface declares only method signatures (and default methods in modern languages) and a class can implement many interfaces. An abstract class can hold state and concrete methods, but a class can extend only one abstract class.",
         "oop", "en"),
        ("Qu'est-ce que l'encapsulation ?",
         "L'encapsulation est le principe consistant a regrouper les donnees et les methodes qui les manipulent dans une classe, et a restreindre l'acces aux donnees internes via des modificateurs de visibilite (prive, protege, public).",
         "oop", "fr"),
        ("Quels sont les principes SOLID ?",
         "SOLID regroupe cinq principes de conception orientee objet : Responsabilite unique, Ouvert/Ferme, Substitution de Liskov, Segregation des interfaces, Inversion des dependances. Ils visent un code maintenable et extensible.",
         "oop", "fr"),
    ],
    "design_patterns.csv": [
        ("What is a design pattern?",
         "A design pattern is a reusable, named solution to a recurring problem in software design. The original GoF catalog groups patterns into creational, structural and behavioral categories.",
         "patterns", "en"),
        ("What is the Singleton pattern?",
         "The Singleton is a creational pattern that ensures a class has at most one instance and provides a global point of access to it. It is useful for shared resources like configuration or logging.",
         "patterns", "en"),
        ("What is the Observer pattern?",
         "The Observer is a behavioral pattern where an object (subject) maintains a list of dependents (observers) and notifies them automatically of any state change, typically by calling one of their methods.",
         "patterns", "en"),
        ("What is the Factory pattern?",
         "The Factory pattern is a creational pattern that delegates object creation to a separate method or class, allowing the calling code to remain decoupled from the concrete classes it instantiates.",
         "patterns", "en"),
        ("What is the Strategy pattern?",
         "The Strategy pattern defines a family of interchangeable algorithms, encapsulates each one, and lets the client choose which algorithm to use at runtime, favoring composition over inheritance.",
         "patterns", "en"),
        ("What is the MVC pattern?",
         "MVC (Model-View-Controller) is an architectural pattern that separates the data and business logic (Model), the user interface (View) and the input/coordination logic (Controller), improving modularity and testability.",
         "patterns", "en"),
    ],
    "git.csv": [
        ("What is Git?",
         "Git is a distributed version control system that tracks changes to files, supports non-linear development through branches and merges, and lets every developer hold a full copy of the project history locally.",
         "git", "en"),
        ("What is a Git branch?",
         "A Git branch is a lightweight, movable pointer to a commit. Branches let developers work on features, bug fixes or experiments in isolation before merging them back into the main line.",
         "git", "en"),
        ("What is the difference between git merge and git rebase?",
         "Both integrate changes between branches. 'git merge' creates a merge commit preserving full history, while 'git rebase' rewrites commits onto a new base producing a linear history, at the cost of changing commit hashes.",
         "git", "en"),
        ("What is a pull request?",
         "A pull request (or merge request) is a code review mechanism on platforms like GitHub or GitLab where a developer proposes changes from one branch to another, allowing teammates to discuss, review and approve before merging.",
         "git", "en"),
        ("What is git stash?",
         "git stash temporarily saves uncommitted local changes in a stack so you can switch context (e.g. checkout another branch) and restore them later with 'git stash pop'.",
         "git", "en"),
        ("Quelle est la difference entre git merge et git rebase ?",
         "Les deux commandes integrent des modifications entre branches. 'git merge' cree un commit de fusion qui preserve l'historique. 'git rebase' reecrit les commits sur une nouvelle base, produisant un historique lineaire mais modifiant les hash.",
         "git", "fr"),
    ],
    "testing.csv": [
        ("What is unit testing?",
         "Unit testing is the practice of writing automated tests that verify the behavior of the smallest pieces of code (typically functions or methods) in isolation, usually by mocking external dependencies.",
         "testing", "en"),
        ("What is integration testing?",
         "Integration testing verifies that several components work correctly together, exercising the interactions between modules, services, databases or external APIs.",
         "testing", "en"),
        ("What is TDD (Test-Driven Development)?",
         "TDD is a development discipline summarized as 'red-green-refactor': write a failing test, write the minimum code to pass it, then refactor while keeping the tests green.",
         "testing", "en"),
        ("What is the difference between mocks and stubs?",
         "Stubs are objects that return canned answers to calls during the test, while mocks are pre-programmed with expectations about which calls they should receive and can fail the test if those expectations are not met.",
         "testing", "en"),
        ("What is code coverage?",
         "Code coverage is a metric that measures the proportion of source code executed by a test suite, expressed as a percentage. Common forms are line, branch and statement coverage.",
         "testing", "en"),
    ],
    "databases.csv": [
        ("What is a relational database?",
         "A relational database organizes data into tables of rows and columns, with relationships expressed through foreign keys, and is queried using SQL. Examples include PostgreSQL, MySQL and SQLite.",
         "db", "en"),
        ("What is normalization?",
         "Normalization is the process of structuring a relational schema to reduce redundancy and avoid update anomalies, typically by decomposing tables and applying normal forms (1NF, 2NF, 3NF, BCNF).",
         "db", "en"),
        ("What is the difference between SQL and NoSQL?",
         "SQL databases enforce a fixed relational schema and ACID transactions; NoSQL databases (document, key-value, column, graph) trade strict schemas for flexibility, horizontal scalability and varied data models.",
         "db", "en"),
        ("What is an index in a database?",
         "An index is a data structure (commonly a B-tree) that speeds up read queries on a table column at the cost of slower writes and extra storage, similar to the index at the back of a book.",
         "db", "en"),
        ("What is ACID?",
         "ACID stands for Atomicity, Consistency, Isolation and Durability - four properties that guarantee reliable processing of database transactions even in the face of errors or crashes.",
         "db", "en"),
    ],
    "apis.csv": [
        ("What is a REST API?",
         "A REST API is an HTTP-based interface that exposes resources identified by URLs and manipulated through a small set of standard verbs (GET, POST, PUT, PATCH, DELETE), typically exchanging JSON, and that is stateless between requests.",
         "api", "en"),
        ("What is the difference between REST and GraphQL?",
         "REST exposes multiple endpoints each returning a fixed shape; GraphQL exposes a single endpoint where the client describes exactly what fields it needs in a query, reducing over-fetching but adding server-side complexity.",
         "api", "en"),
        ("What is an idempotent HTTP method?",
         "An HTTP method is idempotent when calling it multiple times has the same effect as calling it once. GET, PUT and DELETE are idempotent; POST generally is not.",
         "api", "en"),
        ("What is API authentication via JWT?",
         "JWT (JSON Web Token) is a compact, signed token format used for stateless authentication. The server issues a token after login; the client sends it in the Authorization header on subsequent requests for the server to verify.",
         "api", "en"),
    ],
    "devops.csv": [
        ("What is DevOps?",
         "DevOps is a set of cultural practices and tools that bridge software development and IT operations, aiming to shorten the delivery cycle through automation, collaboration, continuous integration and continuous delivery.",
         "devops", "en"),
        ("What is CI/CD?",
         "CI/CD stands for Continuous Integration (automatically building and testing every code change) and Continuous Delivery/Deployment (automatically releasing those changes to staging or production environments).",
         "devops", "en"),
        ("What is Docker?",
         "Docker is a platform that packages an application and its dependencies into a lightweight, portable container that runs identically on any host with a compatible container runtime.",
         "devops", "en"),
        ("What is the difference between a container and a virtual machine?",
         "A virtual machine emulates an entire operating system on top of a hypervisor; a container shares the host kernel and isolates only user space, making containers far lighter, faster to start and more density-friendly.",
         "devops", "en"),
        ("What is Kubernetes?",
         "Kubernetes is an open-source container orchestrator that automates the deployment, scaling, networking and self-healing of containerized applications across a cluster of machines.",
         "devops", "en"),
    ],
    "system_design.csv": [
        ("What is software architecture?",
         "Software architecture is the set of high-level structural decisions about a system: its components, their responsibilities, the interactions between them, and the constraints that shape them.",
         "arch", "en"),
        ("What is the difference between monolithic and microservices architecture?",
         "A monolith deploys all functionality as a single unit; microservices split the system into small, independently deployable services that communicate over the network, trading operational simplicity for scalability and team autonomy.",
         "arch", "en"),
        ("What is coupling and cohesion?",
         "Coupling measures how strongly modules depend on each other (we aim for low coupling); cohesion measures how closely the responsibilities inside a module belong together (we aim for high cohesion).",
         "arch", "en"),
        ("What is load balancing?",
         "Load balancing distributes incoming network traffic across multiple servers to improve responsiveness and availability, using strategies such as round-robin, least connections or hashing.",
         "arch", "en"),
        ("What is caching?",
         "Caching stores the result of expensive computations or remote calls in fast storage so subsequent identical requests can be served quickly, at the cost of staleness and added complexity around invalidation.",
         "arch", "en"),
        ("Difference entre architecture monolithique et microservices ?",
         "Une architecture monolithique deploie toutes les fonctionnalites d'un seul bloc. Les microservices decomposent le systeme en services petits et deployables independamment qui communiquent par le reseau, gagnant en scalabilite au prix d'une complexite operationnelle accrue.",
         "arch", "fr"),
    ],
}

for filename, rows in seed_data.items():
    df = pd.DataFrame(rows, columns=["question", "answer", "category", "language"])
    df.to_csv(DATA_DIR / filename, index=False)

print(f"Wrote {len(seed_data)} CSV files to {DATA_DIR}/:")
for p in sorted(DATA_DIR.glob('*.csv')):
    n = sum(1 for _ in open(p)) - 1
    print(f"  {p.name:<22} ({n} rows)")


## 3. Dataset loading & merging

The loader is **schema-driven**: it scans `data/` for CSVs and concatenates every file with the expected columns. To plug a Stack Exchange dump, simply convert it to that schema and drop it in the folder — no code change required.

The unified DataFrame is the single source of truth for the rest of the pipeline.


In [ ]:
# --- Section 3.1 : Generic CSV loader ------------------------------------
EXPECTED_COLS = {"question", "answer"}      # mandatory
OPTIONAL_COLS = {"category", "language"}    # filled with defaults if missing

def load_all_csvs(data_dir=DATA_DIR):
    # Load every CSV in data_dir that has at least question + answer columns.
    frames = []
    for csv_path in sorted(data_dir.glob("*.csv")):
        df = pd.read_csv(csv_path)
        if not EXPECTED_COLS.issubset(df.columns):
            print(f"  Skipping {csv_path.name}: missing required columns")
            continue
        if "category" not in df.columns:
            df["category"] = csv_path.stem
        if "language" not in df.columns:
            df["language"] = "en"
        df["source_file"] = csv_path.name
        frames.append(df[["question", "answer", "category", "language", "source_file"]])
        print(f"  Loaded {csv_path.name:<22} -> {len(df):>4} rows")
    if not frames:
        raise FileNotFoundError(f"No valid CSV found in {data_dir}")
    return pd.concat(frames, ignore_index=True)

raw_df = load_all_csvs()
print(f"\nTotal raw rows : {len(raw_df)}")
print(f"Categories     : {sorted(raw_df['category'].unique())}")
print(f"Languages      : {raw_df['language'].value_counts().to_dict()}")
raw_df.head(3)


## 4. Preprocessing pipeline + data augmentation

The cleaner is a small composable chain so each step is testable in isolation:

1. lowercase
2. strip accents (FR diacritics → ASCII for typo robustness on the *retrieval key*; the original casing is kept for display)
3. remove redundant punctuation
4. normalize whitespace

We then **augment** every question with light paraphrase variants (lead-in stems) to widen the retrieval surface without inflating the dataset artificially.


In [ ]:
# --- Section 4.1 : Text cleaner -------------------------------------------
from unidecode import unidecode

_PUNCT_RE = re.compile(r"[^\w\s\?\!\.\,\-:;/]")
_WS_RE    = re.compile(r"\s+")

def clean_text(text, drop_accents=True):
    if not isinstance(text, str):
        return ""
    t = text.strip().lower()
    if drop_accents:
        t = unidecode(t)
    t = _PUNCT_RE.sub(" ", t)
    t = _WS_RE.sub(" ", t).strip()
    return t

# Quick sanity check
samples = [
    "  Qu'est-ce que l'ENCAPSULATION ??  ",
    "What is a REST API?",
    "C'est quoi le couplage faible vs fort ???",
]
for s in samples:
    print(f"{s!r:<55} ->  {clean_text(s)!r}")


In [ ]:
# --- Section 4.2 : Light data augmentation --------------------------------
EN_STEMS = [
    "{q}",
    "Can you explain: {q}",
    "I want to understand {q}",
    "Could you describe {q}",
    "In software engineering, {q}",
]
FR_STEMS = [
    "{q}",
    "Peux-tu m'expliquer {q}",
    "J'aimerais comprendre {q}",
    "Explique-moi {q}",
    "En genie logiciel, {q}",
]

def augment_question(q, language):
    stems = FR_STEMS if language == "fr" else EN_STEMS
    q_low = q.rstrip("?.! ").lower()
    return [stems[0].format(q=q)] + [s.format(q=q_low) for s in stems[1:]]

# --- Section 4.3 : Build the corpus ---------------------------------------
records = []
for _, row in raw_df.iterrows():
    for variant in augment_question(row["question"], row["language"]):
        records.append({
            "question_raw":  variant,
            "question_norm": clean_text(variant),
            "answer":        row["answer"],
            "category":      row["category"],
            "language":      row["language"],
            "source_file":   row["source_file"],
        })

corpus_df = pd.DataFrame(records).drop_duplicates(subset=["question_norm"]).reset_index(drop=True)
print(f"Raw rows         : {len(raw_df)}")
print(f"Augmented rows   : {len(corpus_df)}  (~{len(corpus_df)/len(raw_df):.1f}x)")
print(f"Per language     : {corpus_df['language'].value_counts().to_dict()}")
corpus_df.sample(5, random_state=0)


## 5. Multilingual sentence embeddings

We use **`paraphrase-multilingual-MiniLM-L12-v2`** (≈ 120 MB, 384-dim, supports 50+ languages including French). This single model means a French question and its English answer live in the same vector space — no translation step needed for retrieval itself.

Embeddings are L2-normalized so that inner product on the FAISS index is equivalent to cosine similarity.


In [ ]:
# --- Section 5.1 : Load the encoder ---------------------------------------
EMBED_MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

print(f"Loading encoder: {EMBED_MODEL_NAME}")
encoder = SentenceTransformer(EMBED_MODEL_NAME, device=DEVICE)
print(f"Embedding dim   : {encoder.get_sentence_embedding_dimension()}")


In [ ]:
# --- Section 5.2 : Encode the whole corpus --------------------------------
def embed(texts, batch_size=64):
    vecs = encoder.encode(
        texts,
        batch_size=batch_size,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,        # L2-norm so dot product == cosine
    )
    return vecs.astype("float32")

corpus_vecs = embed(corpus_df["question_norm"].tolist())
print(f"\nCorpus embeddings shape: {corpus_vecs.shape}")

np.save(ARTIFACT_DIR / "corpus_vecs.npy", corpus_vecs)
corpus_df.to_parquet(ARTIFACT_DIR / "corpus.parquet")
print(f"Saved to {ARTIFACT_DIR}/")


## 6. FAISS index

For a corpus of this size a flat index (`IndexFlatIP`) is exact and instant. The same code scales to ~1 M vectors; beyond that, switch to `IndexHNSWFlat` or `IndexIVFFlat` — only the index constructor changes, the rest of the pipeline is identical.


In [ ]:
# --- Section 6.1 : Build the index ----------------------------------------
def build_faiss_index(vectors):
    dim = vectors.shape[1]
    index = faiss.IndexFlatIP(dim)        # inner product on normalized vectors == cosine
    index.add(vectors)
    return index

faiss_index = build_faiss_index(corpus_vecs)
print(f"FAISS index ntotal : {faiss_index.ntotal}")
print(f"FAISS index dim    : {faiss_index.d}")

faiss.write_index(faiss_index, str(INDEX_DIR / "corpus.faiss"))
print(f"Saved index to {INDEX_DIR}/corpus.faiss")


In [ ]:
# --- Section 6.2 : Smoke-test retrieval -----------------------------------
def search_topk(query, k=5):
    q_vec = embed([clean_text(query)])
    scores, idxs = faiss_index.search(q_vec, k)
    out = []
    for score, i in zip(scores[0], idxs[0]):
        row = corpus_df.iloc[int(i)]
        out.append({
            "score":    float(score),
            "question": row["question_raw"],
            "answer":   row["answer"],
            "category": row["category"],
            "language": row["language"],
        })
    return out

for q in ["What is Scrum?", "Explique-moi le couplage", "Difference between SQL and NoSQL"]:
    print(f"\nQ: {q}")
    for hit in search_topk(q, k=3):
        print(f"   [{hit['score']:.3f}] ({hit['category']}) {hit['question'][:70]}")


## 7. Cross-encoder reranking

Bi-encoders (Section 5) are fast but encode query and document **independently**. A cross-encoder reads them **together** and scores their relevance directly, which is more accurate but slower — so we use it only on the Top-K candidates from FAISS, not on the full corpus.

`cross-encoder/ms-marco-MiniLM-L-6-v2` is the standard lightweight choice (~80 MB).


In [ ]:
# --- Section 7.1 : Load the reranker --------------------------------------
RERANKER_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"
print(f"Loading reranker: {RERANKER_NAME}")
reranker = CrossEncoder(RERANKER_NAME, device=DEVICE)
print("Ready.")


In [ ]:
# --- Section 7.2 : Retrieve + rerank -------------------------------------
def retrieve_and_rerank(query, k_retrieve=20, k_final=5):
    candidates = search_topk(query, k=k_retrieve)
    pairs = [(query, c["question"]) for c in candidates]
    ce_scores = reranker.predict(pairs, show_progress_bar=False)
    for c, s in zip(candidates, ce_scores):
        c["rerank_score"] = float(s)
    candidates.sort(key=lambda x: x["rerank_score"], reverse=True)
    return candidates[:k_final]

print("Reranked Top-3 for: 'How do I write a unit test?'")
for h in retrieve_and_rerank("How do I write a unit test?", k_final=3):
    print(f"   bi={h['score']:.3f}  ce={h['rerank_score']:+.2f}  {h['question'][:70]}")


## 8. Language detection & optional translation

Because the generator (FLAN-T5) is mainly English-pretrained, we:

* **detect** the user's language with `langdetect`
* if French, translate the query to English with `Helsinki-NLP/opus-mt-fr-en`
* generate the answer in English
* translate back with `Helsinki-NLP/opus-mt-en-fr`

Translation models are loaded **lazily** so an English-only session never pays the cost.


In [ ]:
# --- Section 8.1 : Language detection ------------------------------------
from langdetect import detect, DetectorFactory
DetectorFactory.seed = 0      # deterministic

def detect_language(text):
    try:
        lang = detect(text)
        return "fr" if lang == "fr" else "en"
    except Exception:
        return "en"

for s in ["What is OOP?", "Qu'est-ce que la POO ?", "Explique-moi REST"]:
    print(f"{s!r:<35} -> {detect_language(s)}")


In [ ]:
# --- Section 8.2 : Lazy-loaded translation pipelines ---------------------
from transformers import pipeline

_translators = {}

def get_translator(direction):
    # direction in {'fr-en', 'en-fr'}
    if direction not in _translators:
        model_name = f"Helsinki-NLP/opus-mt-{direction}"
        print(f"Loading translator: {model_name}")
        _translators[direction] = pipeline(
            "translation",
            model=model_name,
            device=0 if DEVICE == "cuda" else -1,
        )
    return _translators[direction]

def translate(text, direction):
    if not text.strip():
        return text
    tr = get_translator(direction)
    return tr(text, max_length=512)[0]["translation_text"]

# Smoke test (loads the model on first call, will take a few seconds)
print(translate("Qu'est-ce que le couplage faible ?", "fr-en"))


## 9. FLAN-T5 generator with conversational memory

The retrieved + reranked contexts are concatenated into a prompt along with a rolling chat history (last *N* turns). FLAN-T5-base is small enough for Colab CPU; swap for `google/flan-t5-large` if you have a GPU.

A **confidence threshold** on the top reranker score decides whether to:

* **answer** with the generator grounded on the retrieved context
* or **politely refuse** ("I don't have a confident answer in my knowledge base") to avoid hallucinations.


In [ ]:
# --- Section 9.1 : Load the generator ------------------------------------
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

GEN_MODEL_NAME = "google/flan-t5-base"
print(f"Loading generator: {GEN_MODEL_NAME}")
gen_tokenizer = AutoTokenizer.from_pretrained(GEN_MODEL_NAME)
gen_model     = AutoModelForSeq2SeqLM.from_pretrained(GEN_MODEL_NAME).to(DEVICE)
gen_model.eval()
print("Ready.")


In [ ]:
# --- Section 9.2 : Prompt + generation -----------------------------------
PROMPT_TEMPLATE = (
    "You are a software engineering tutor. Use ONLY the context to answer the "
    "question concisely and accurately. If the context does not contain the "
    "answer, say you do not know.\n\n"
    "Context:\n{context}\n\n"
    "{history}Question: {question}\nAnswer:"
)

def build_prompt(question, contexts, history):
    ctx_block = "\n".join(
        f"- ({c['category']}) Q: {c['question']}\n  A: {c['answer']}"
        for c in contexts
    )
    hist_block = ""
    if history:
        hist_block = "Previous conversation:\n" + "\n".join(
            f"User: {u}\nAssistant: {a}" for u, a in history
        ) + "\n\n"
    return PROMPT_TEMPLATE.format(context=ctx_block, history=hist_block, question=question)

@torch.no_grad()
def generate_answer(prompt, max_new_tokens=180):
    inputs = gen_tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024).to(DEVICE)
    out = gen_model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        num_beams=4,
        do_sample=False,
        early_stopping=True,
    )
    return gen_tokenizer.decode(out[0], skip_special_tokens=True).strip()


## 10. End-to-end RAG pipeline

`SoftwareEngineeringChatbot` is the public entry point that ties Sections 4–9 together. It also keeps a per-session memory of the last few turns.


In [ ]:
# --- Section 10.1 : Pipeline orchestrator --------------------------------
class SoftwareEngineeringChatbot:
    def __init__(
        self,
        k_retrieve=20,
        k_final=4,
        threshold=1.0,        # CrossEncoder logit threshold; tune in Section 11
        memory_turns=3,
        translate_fr=True,
    ):
        self.k_retrieve   = k_retrieve
        self.k_final      = k_final
        self.threshold    = threshold
        self.memory_turns = memory_turns
        self.translate_fr = translate_fr
        self.history      = []   # list of (user_q, assistant_a) tuples

    def reset(self):
        self.history = []

    def answer(self, user_question):
        lang = detect_language(user_question)
        query_en = user_question
        translated_in = False
        if lang == "fr" and self.translate_fr:
            query_en = translate(user_question, "fr-en")
            translated_in = True

        # Retrieval + rerank operate on the (possibly translated) English query
        candidates = retrieve_and_rerank(query_en, self.k_retrieve, self.k_final)
        top_score  = candidates[0]["rerank_score"] if candidates else -1e9

        if top_score < self.threshold:
            answer_text = (
                "I don't have a confident answer in my current knowledge base. "
                "Could you rephrase or ask about another software engineering topic?"
            )
            refused = True
        else:
            prompt = build_prompt(query_en, candidates, self.history[-self.memory_turns:])
            answer_text = generate_answer(prompt)
            refused = False

        if lang == "fr" and self.translate_fr and not refused:
            answer_text_user = translate(answer_text, "en-fr")
        else:
            answer_text_user = answer_text

        self.history.append((user_question, answer_text_user))

        return {
            "answer":         answer_text_user,
            "answer_english": answer_text,
            "language":       lang,
            "translated_in":  translated_in,
            "top_score":      top_score,
            "refused":        refused,
            "candidates":     candidates,
        }

bot = SoftwareEngineeringChatbot(threshold=1.0)

# Smoke tests
for q in [
    "What is the Observer design pattern?",
    "Quelle est la difference entre agregation et composition ?",
    "How do I cook spaghetti carbonara?",   # out of domain -> should refuse
]:
    res = bot.answer(q)
    flag = "REFUSED" if res["refused"] else "ANSWERED"
    print(f"\n[{flag}] (top={res['top_score']:+.2f}, lang={res['language']})")
    print(f"Q: {q}")
    print(f"A: {res['answer']}")


## 11. Evaluation

We measure two things:

| Metric                  | What it tells us                                                    |
|-------------------------|---------------------------------------------------------------------|
| **Retrieval Accuracy@k**| Probability that the correct source row is among the Top-k          |
| **MRR (Mean Reciprocal Rank)** | How high the correct row ranks on average                    |
| **Semantic similarity** | Cosine similarity between generated answer and gold answer          |
| **Confusion examples**  | A handful of queries where the system is wrong, for error analysis  |

We hold out 20% of the **original** rows (before augmentation) as a test set.


In [ ]:
# --- Section 11.1 : Build a held-out test set -----------------------------
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    raw_df, test_size=0.2, random_state=42, stratify=raw_df["category"]
)
print(f"Train rows : {len(train_df)}")
print(f"Test rows  : {len(test_df)}")
test_df.head(3)


In [ ]:
# --- Section 11.2 : Retrieval metrics ------------------------------------
def evaluate_retrieval(test_df, ks=(1, 3, 5)):
    hits = {k: 0 for k in ks}
    rr_sum = 0.0
    for _, row in test_df.iterrows():
        q = row["question"]
        gold_answer = row["answer"]
        results = retrieve_and_rerank(q, k_retrieve=20, k_final=max(ks))
        # 'correct' = retrieved row whose answer matches the gold answer
        rank = next((i for i, r in enumerate(results, 1) if r["answer"] == gold_answer), None)
        if rank is not None:
            rr_sum += 1.0 / rank
            for k in ks:
                if rank <= k:
                    hits[k] += 1
    n = len(test_df)
    return {
        **{f"Accuracy@{k}": hits[k] / n for k in ks},
        "MRR": rr_sum / n,
        "n_test": n,
    }

retrieval_metrics = evaluate_retrieval(test_df)
print("Retrieval metrics:")
for k, v in retrieval_metrics.items():
    if isinstance(v, float):
        print(f"  {k:<14}: {v:.4f}")
    else:
        print(f"  {k:<14}: {v}")


In [ ]:
# --- Section 11.3 : Generation quality (semantic similarity to gold) -----
from sklearn.metrics.pairwise import cosine_similarity

def evaluate_generation(test_df, sample_size=15):
    sample = test_df.sample(min(sample_size, len(test_df)), random_state=0)
    sims, rows = [], []
    bot_eval = SoftwareEngineeringChatbot(threshold=-1e9, translate_fr=False)  # never refuse, for eval
    for _, row in sample.iterrows():
        pred = bot_eval.answer(row["question"])["answer_english"]
        gold = row["answer"]
        v_pred = embed([clean_text(pred)])
        v_gold = embed([clean_text(gold)])
        sim = float(cosine_similarity(v_pred, v_gold)[0, 0])
        sims.append(sim)
        rows.append({"question": row["question"], "gold": gold[:80], "pred": pred[:80], "sim": sim})
    return {
        "avg_semantic_similarity": float(np.mean(sims)),
        "samples": pd.DataFrame(rows).sort_values("sim"),
    }

gen_metrics = evaluate_generation(test_df, sample_size=10)
print(f"Avg semantic similarity (pred vs gold): {gen_metrics['avg_semantic_similarity']:.4f}\n")
print("Worst examples (for error analysis):")
gen_metrics["samples"].head(5)


In [ ]:
# --- Section 11.4 : Confusion examples ------------------------------------
# Cases where the user's intent and the top retrieved item differ -- useful to grow the dataset.
confusion = []
for _, row in test_df.iterrows():
    results = retrieve_and_rerank(row["question"], k_final=1)
    top = results[0]
    if top["answer"] != row["answer"]:
        confusion.append({
            "question": row["question"],
            "expected_category": row["category"],
            "got_category": top["category"],
            "got_question": top["question"],
            "score": top["rerank_score"],
        })

confusion_df = pd.DataFrame(confusion)
print(f"Confusion cases: {len(confusion_df)} / {len(test_df)}")
confusion_df.head(10) if len(confusion_df) else "No confusion cases."


## 12. Gradio chat interface

A simple but production-flavored UI:

* multi-turn chat (uses the same `SoftwareEngineeringChatbot` instance, so memory persists across turns)
* threshold slider (live)
* debug panel showing detected language, top score, and retrieved candidates
* "Reset conversation" button to clear memory


In [ ]:
# --- Section 12.1 : Gradio app -------------------------------------------
import gradio as gr

ui_bot = SoftwareEngineeringChatbot(threshold=1.0)

def chat_fn(message, history_ui, threshold, translate_fr):
    ui_bot.threshold = threshold
    ui_bot.translate_fr = translate_fr
    res = ui_bot.answer(message)
    debug = (
        f"Detected language : {res['language']}\n"
        f"Top rerank score  : {res['top_score']:+.3f}\n"
        f"Refused           : {res['refused']}\n"
        f"Top candidates    :\n" +
        "\n".join(f"  [{c['rerank_score']:+.2f}] ({c['category']}) {c['question'][:70]}"
                  for c in res["candidates"])
    )
    history_ui = history_ui + [(message, res["answer"])]
    return history_ui, "", debug

def reset_fn():
    ui_bot.reset()
    return [], "", "Conversation memory cleared."

with gr.Blocks(theme=gr.themes.Soft(), title="Software Engineering RAG Chatbot") as demo:
    gr.Markdown(
        "# 🤖 Software Engineering RAG Chatbot\n"
        "Ask in **English** or **French** about UML, OOP, design patterns, Git, "
        "testing, databases, APIs, DevOps, system design, Agile."
    )

    with gr.Row():
        with gr.Column(scale=3):
            chatbot_widget = gr.Chatbot(label="Conversation", height=420)
            msg = gr.Textbox(
                label="Your question",
                placeholder="Ex: Quelle est la difference entre REST et GraphQL ?",
            )
            with gr.Row():
                send  = gr.Button("Send", variant="primary")
                reset = gr.Button("Reset conversation")
        with gr.Column(scale=2):
            threshold = gr.Slider(
                -5.0, 10.0, value=1.0, step=0.5,
                label="Refusal threshold (higher = stricter)",
            )
            translate_fr = gr.Checkbox(value=True, label="Translate FR <-> EN through pipeline")
            debug = gr.Textbox(label="Debug", lines=12)

    gr.Examples(
        examples=[
            "What is the Observer design pattern?",
            "Difference between aggregation and composition?",
            "Quels sont les principes SOLID ?",
            "Explique-moi git rebase vs git merge",
            "What is CI/CD?",
            "How do I make a pizza?",   # out of domain
        ],
        inputs=msg,
    )

    send.click(chat_fn, [msg, chatbot_widget, threshold, translate_fr],
               [chatbot_widget, msg, debug])
    msg.submit(chat_fn, [msg, chatbot_widget, threshold, translate_fr],
               [chatbot_widget, msg, debug])
    reset.click(reset_fn, outputs=[chatbot_widget, msg, debug])

# Launch with share=True so the link works on Colab/Kaggle.
demo.launch(share=True, debug=False)


## ✅ What you have now

| Requirement                               | Where it lives          |
|-------------------------------------------|-------------------------|
| External CSV datasets                     | Section 2 + Section 3   |
| Modular preprocessing + augmentation      | Section 4               |
| Multilingual embeddings                   | Section 5               |
| FAISS index (scalable)                    | Section 6               |
| Top-K retrieval + CrossEncoder reranking  | Section 7               |
| Language detection + FR/EN translation    | Section 8               |
| T5 generation + conversational memory     | Sections 9 and 10       |
| Confidence threshold (no hallucinations)  | Section 10              |
| Retrieval Accuracy@k, MRR, semantic sim   | Section 11              |
| Gradio multi-turn UI                      | Section 12              |

### Plugging in real datasets

Convert any of these into CSVs with columns `question,answer,category,language` and drop them into `data/`:

* Stack Exchange dump → `softwareengineering.stackexchange.com.7z` → parse `Posts.xml`
* Kaggle StackSample (60k programming Q&A)
* Hugging Face: `flax-sentence-embeddings/stackexchange_xml`, `mteb/stackexchange-clustering`
* Your own course notes converted to Q/A pairs

Then re-run Sections 3 → 6 and the entire pipeline picks them up automatically.

### Next improvements

1. **Hard-negative mining + fine-tune the bi-encoder** on your own Q-A pairs (`MultipleNegativesRankingLoss`).
2. **Hybrid retrieval**: BM25 (`rank_bm25`) ∪ FAISS, then rerank — much more robust to rare terms.
3. **Streaming generation** in the UI (`TextIteratorStreamer`).
4. **Per-user persistent memory** (SQLite/Redis) instead of in-process history.
5. **Larger generator**: swap `flan-t5-base` for `flan-t5-large` or a 7B instruct model on GPU.
